# cryp2me.ai — Phase 2: ML Training Notebook

This notebook walks through the full Phase 2 pipeline step by step.
Use this for experimentation, visualization, and understanding what each stage does.
For production training, run `python scripts/train.py` instead.

## What this notebook covers:
1. Data download & inspection
2. Feature engineering visualization
3. Dataset construction
4. LSTM training + loss curves
5. Transformer training + comparison
6. Ensemble + threshold analysis
7. Accuracy report
8. ONNX export

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import torch
from pathlib import Path

from configs.config import cfg

device = cfg.resolve_device()
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')
print(f'Features: {cfg.features.n_features} per timestep')
print(f'Sequence length: {cfg.data.sequence_length} days')

## Step 1 — Download Data

Downloads OHLCV from Binance and caches to `data/cache/`. Subsequent runs use the cache.

In [ ]:
from src.data.collector import collect_all

# Use a small set for notebook exploration
TICKERS = ['BTC', 'ETH', 'SOL', 'BNB', 'XRP']

raw_data = collect_all(
    TICKERS,
    interval='1d',
    lookback_days=4 * 365,
    cache_dir=cfg.data.cache_dir,
)

# Inspect BTC
btc = raw_data['BTC']
print(f'BTC: {len(btc)} candles')
print(f'From: {pd.to_datetime(btc.time.min(), unit="s").date()}')
print(f'To:   {pd.to_datetime(btc.time.max(), unit="s").date()}')
btc.tail()

## Step 2 — Feature Engineering

Compute all 18 technical features. Visualize to make sure they look correct.

In [ ]:
from src.data.features import build_features, build_all

btc_feat = build_features(btc)
print(f'Features: {btc_feat.shape}')
print('Columns:', list(btc_feat.columns))
btc_feat.describe().round(4)

In [ ]:
# Visualize BTC with indicators — sanity check
dates = pd.to_datetime(btc_feat['time'], unit='s')

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
fig.suptitle('BTC — Feature Sanity Check', fontsize=14, fontweight='bold')

# Price + EMAs
close = raw_data['BTC']['close'].values[-len(btc_feat):]
axes[0].plot(dates, close, color='#1e3a5f', linewidth=1, label='Close')
axes[0].set_title('Close Price')
axes[0].legend()

# EMA distances
axes[1].plot(dates, btc_feat['ema10_dist'], color='#4dabff', alpha=0.8, label='EMA10 dist')
axes[1].plot(dates, btc_feat['ema20_dist'], color='#a78bfa', alpha=0.8, label='EMA20 dist')
axes[1].plot(dates, btc_feat['ema34_dist'], color='#fb923c', alpha=0.8, label='EMA34 dist')
axes[1].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_title('EMA Distance from Close')
axes[1].legend()

# RSI
axes[2].plot(dates, btc_feat['rsi14'] * 100, color='#60a5fa', label='RSI(14)')
axes[2].axhline(70, color='#ff4d6a', linestyle='--', alpha=0.7, label='OB')
axes[2].axhline(30, color='#00d49a', linestyle='--', alpha=0.7, label='OS')
axes[2].set_ylim(0, 100)
axes[2].set_title('RSI(14)')
axes[2].legend()

# MACD
hist = btc_feat['macd_hist'] * close  # denormalise for visibility
colors = ['#00d49a' if v >= 0 else '#ff4d6a' for v in hist]
axes[3].bar(dates, hist, color=colors, alpha=0.7, width=1.5, label='Histogram')
axes[3].plot(dates, btc_feat['macd'] * close, color='#e2e8f0', linewidth=1, label='MACD')
axes[3].plot(dates, btc_feat['macd_signal'] * close, color='#f472b6', linewidth=1, label='Signal')
axes[3].set_title('MACD')
axes[3].legend()

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
    ax.grid(alpha=0.15)

plt.tight_layout()
plt.savefig('../reports/btc_features_check.png', dpi=120, bbox_inches='tight')
plt.show()
print('✓ Features look correct')

## Step 3 — Dataset Construction

Build sliding window sequences. Each sample is (seq_len=60, n_features=18).

In [ ]:
from src.data.features import build_all
from src.data.dataset import CryptoSequenceDataset, split_df_chronological
from torch.utils.data import DataLoader

all_features = build_all(raw_data)

train_dfs, val_dfs = [], []
for t, df in all_features.items():
    tr, vl = split_df_chronological(df, val_ratio=0.15)
    train_dfs.append(tr)
    val_dfs.append(vl)

train_ds = CryptoSequenceDataset(train_dfs, seq_len=60, augment=True)
val_ds   = CryptoSequenceDataset(val_dfs,   seq_len=60, augment=False)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False)

print(f'Train samples: {len(train_ds):,}')
print(f'Val samples:   {len(val_ds):,}')

# Inspect a batch
x, y_reg, y_cls, price = next(iter(train_loader))
print(f'x shape:     {x.shape}   (batch, seq_len, n_features)')
print(f'y_reg shape: {y_reg.shape} (batch, 3 horizons)')
print(f'y_cls shape: {y_cls.shape} (batch, 3 horizons)')
print(f'Label balance: {y_cls[:, 0].mean():.3f} (should be ~0.50)')

## Step 4 — LSTM Training

Train the LSTM model and plot loss + accuracy curves.

In [ ]:
from src.models.lstm_model import LSTMModel
from src.training.trainer import train_model

lstm = LSTMModel(n_features=18, hidden_sizes=[256, 128, 64], dropout=0.2)
print(f'LSTM parameters: {lstm.n_params:,}')

lstm_history = train_model(
    lstm, train_loader, val_loader, device,
    epochs=50,        # increase to 100 for production
    lr=1e-3,
    patience=10,
    model_name='LSTM'
)

In [ ]:
# Plot LSTM training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(lstm_history['train_loss'], label='Train', color='#4dabff')
ax1.plot(lstm_history['val_loss'],   label='Val',   color='#fb923c')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('LSTM — Loss')
ax1.legend(); ax1.grid(alpha=0.2)

accs = [a * 100 for a in lstm_history['val_dir_acc']]
ax2.plot(accs, color='#00d49a', linewidth=2)
ax2.axhline(84, color='#ff4d6a', linestyle='--', alpha=0.7, label='84% target')
ax2.axhline(50, color='gray',    linestyle='--', alpha=0.5, label='Baseline')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy %'); ax2.set_title('LSTM — Directional Accuracy')
ax2.legend(); ax2.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('../reports/lstm_training.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Best LSTM val dir_acc: {max(accs):.2f}%')

## Step 5 — Transformer Training

In [ ]:
from src.models.transformer_model import TransformerModel

transformer = TransformerModel(
    n_features=18, d_model=128, nhead=8,
    num_layers=4, dim_feedforward=512, dropout=0.1
)
print(f'Transformer parameters: {transformer.n_params:,}')

tf_history = train_model(
    transformer, train_loader, val_loader, device,
    epochs=50,
    lr=1e-3,
    patience=10,
    model_name='Transformer'
)

In [ ]:
# Compare both models
fig, ax = plt.subplots(figsize=(11, 5))

lstm_accs = [a * 100 for a in lstm_history['val_dir_acc']]
tf_accs   = [a * 100 for a in tf_history['val_dir_acc']]

ax.plot(lstm_accs, label='LSTM',        color='#4dabff', linewidth=2)
ax.plot(tf_accs,   label='Transformer', color='#a78bfa', linewidth=2)
ax.axhline(84, color='#ff4d6a', linestyle='--', alpha=0.8, label='84% Target')
ax.axhline(50, color='gray',    linestyle=':', alpha=0.5, label='Baseline')

ax.set_xlabel('Epoch'); ax.set_ylabel('Validation Directional Accuracy (%)')
ax.set_title('LSTM vs Transformer — Directional Accuracy')
ax.legend(); ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('../reports/model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Best LSTM:        {max(lstm_accs):.2f}%')
print(f'Best Transformer: {max(tf_accs):.2f}%')

## Step 6 — Ensemble + Threshold Analysis

Combine both models and find the confidence threshold that achieves 84%+.

In [ ]:
from src.models.ensemble import MetaLearner
from src.evaluation.metrics import precision_at_threshold
import torch

# Get predictions from both models on val set
lstm.eval(); transformer.eval()
all_lstm_probs, all_tf_probs, all_labels = [], [], []

with torch.no_grad():
    for x, _, y_cls, _ in val_loader:
        x = x.to(device)
        all_lstm_probs.append(lstm.predict_proba(x).cpu().numpy())
        all_tf_probs.append(transformer.predict_proba(x).cpu().numpy())
        all_labels.append(y_cls.numpy())

lstm_probs = np.concatenate(all_lstm_probs, axis=0)
tf_probs   = np.concatenate(all_tf_probs,   axis=0)
labels     = np.concatenate(all_labels,     axis=0)

# Simple average ensemble
ensemble_probs = (lstm_probs + tf_probs) / 2.0

# Threshold sweep
thresholds = np.arange(0.50, 0.85, 0.01)
results = []
for t in thresholds:
    r = precision_at_threshold(ensemble_probs[:, 0], labels[:, 0], threshold=t)
    r['threshold'] = t
    results.append(r)

df_thresh = pd.DataFrame(results)

# Plot precision vs coverage tradeoff
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(df_thresh['threshold'] * 100, df_thresh['precision'] * 100, color='#00d49a', linewidth=2)
ax1.axhline(84, color='#ff4d6a', linestyle='--', alpha=0.7, label='84% target')
ax1.fill_between(df_thresh['threshold'] * 100, df_thresh['precision'] * 100, 84, 
                  where=df_thresh['precision'] >= 0.84, alpha=0.2, color='#00d49a', label='84%+ zone')
ax1.set_xlabel('Confidence Threshold (%)'); ax1.set_ylabel('Precision (%)')
ax1.set_title('Precision vs Threshold')
ax1.legend(); ax1.grid(alpha=0.2)

ax2.plot(df_thresh['coverage'] * 100, df_thresh['precision'] * 100, color='#4dabff', linewidth=2)
ax2.axhline(84, color='#ff4d6a', linestyle='--', alpha=0.7, label='84% target')
ax2.set_xlabel('Coverage (% of samples that get a signal)')
ax2.set_ylabel('Precision (%)')
ax2.set_title('Precision vs Coverage (Tradeoff)')
ax2.legend(); ax2.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('../reports/threshold_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

# Find recommended threshold
high_precision = df_thresh[df_thresh['precision'] >= 0.84]
if len(high_precision) > 0:
    best = high_precision.iloc[0]
    print(f'✅ 84% target achieved at threshold={best["threshold"]:.0%}')
    print(f'   Coverage: {best["coverage"]*100:.1f}% of predictions')
    print(f'   Precision: {best["precision"]*100:.1f}%')
else:
    best = df_thresh.sort_values('precision', ascending=False).iloc[0]
    print(f'⚠ Best precision: {best["precision"]*100:.1f}% at threshold={best["threshold"]:.0%}')
    print('  Consider more training data or feature tuning')

df_thresh[df_thresh['threshold'].isin([0.55, 0.60, 0.65, 0.70, 0.75, 0.80])]

## Step 7 — ONNX Export

Export trained models to ONNX for fast production inference.

In [ ]:
from src.models.ensemble import MetaLearner
from src.export.onnx_export import export_all_models
import torch
import torch.optim as optim

# Train a quick meta-learner
meta = MetaLearner(n_horizons=3).to(device)
optimizer = optim.Adam(meta.parameters(), lr=5e-4)
criterion = torch.nn.BCELoss()

X_lstm = torch.tensor(lstm_probs, dtype=torch.float32).to(device)
X_tf   = torch.tensor(tf_probs,   dtype=torch.float32).to(device)
y      = torch.tensor(labels,     dtype=torch.float32).to(device)

for epoch in range(300):
    optimizer.zero_grad()
    loss = criterion(meta(X_lstm, X_tf), y)
    loss.backward()
    optimizer.step()

print(f'Meta-learner trained, final loss: {loss.item():.4f}')

# Export
onnx_paths = export_all_models(
    lstm, transformer, meta,
    seq_len=60, n_features=18,
    onnx_dir=Path('../models/onnx')
)
print('ONNX models saved to:', onnx_paths)

## Step 8 — Integration Instructions

Models are trained and exported. To wire them into the backend:

```bash
# 1. Copy ONNX models to backend
cp ../models/onnx/*.onnx ../../cryp2me-backend/models/onnx/

# 2. Copy inference service
cp src/export/inference_service.py ../../cryp2me-backend/app/services/

# 3. Replace predict router
cp src/export/predict_router_v2.py ../../cryp2me-backend/app/routers/predict.py

# 4. Install onnxruntime in backend
cd ../../cryp2me-backend && pip install onnxruntime

# 5. Add to backend requirements.txt
echo 'onnxruntime>=1.18.0' >> requirements.txt

# 6. Restart backend
uvicorn app.main:app --reload
```

The frontend needs **zero changes** — it already calls `/api/predict/{ticker}` and renders whatever the backend returns.